In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

In [2]:
# clean data and split manifest load
CURRENT_DIR = Path.cwd()
if CURRENT_DIR.name == "notebooks":
    PROJECT_ROOT = CURRENT_DIR.parent
else:
    PROJECT_ROOT = CURRENT_DIR

DATA_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "telco_churn_clean.csv"
)

MANIFEST_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "split_manifest.csv"
)

df = pd.read_csv(DATA_PATH)
manifest = pd.read_csv(MANIFEST_PATH)
print("Data shape:", df.shape)
print("Manifest shape:", manifest.shape)

Data shape: (7043, 21)
Manifest shape: (7043, 3)


#### merge dataset with Manifest

In [3]:
data = df.merge(
    manifest[["customerID", "split"]],
    on="customerID",
    how="left",
    validate="one_to_one"
)

In [4]:
# check
data["split"].value_counts()

split
train    5634
test     1409
Name: count, dtype: int64

In [5]:
data["split"].isna().sum()

np.int64(0)

#### X_train, X_test, y_train, y_test recreate

In [7]:
train_data = data[data["split"] == "train"].copy()
test_data = data[data["split"] == "test"].copy()

In [8]:
FEATURES_TO_DROP = [
    "customerID",
    "Churn",
    "split"
]

X_train = train_data.drop(
    columns=FEATURES_TO_DROP
)

X_test = test_data.drop(
    columns=FEATURES_TO_DROP
)

In [9]:
# target
y_train = train_data["Churn"].map({
    "No": 0,
    "Yes": 1
})

y_test = test_data["Churn"].map({
    "No": 0,
    "Yes": 1
})

In [10]:
#check
print("X_train:", X_train.shape)
print("X_test :", X_test.shape)

print("y_train:", y_train.shape)
print("y_test :", y_test.shape)

X_train: (5634, 19)
X_test : (1409, 19)
y_train: (5634,)
y_test : (1409,)


#### Feature groups explicitly define

In [11]:
numeric_features = [
    "tenure",
    "MonthlyCharges",
    "TotalCharges"
]

In [12]:
# categorical
categorical_features = [
    "gender",
    "SeniorCitizen",
    "Partner",
    "Dependents",
    "PhoneService",
    "MultipleLines",
    "InternetService",
    "OnlineSecurity",
    "OnlineBackup",
    "DeviceProtection",
    "TechSupport",
    "StreamingTV",
    "StreamingMovies",
    "Contract",
    "PaperlessBilling",
    "PaymentMethod"
]

In [13]:
#check
print("Numerical features:", len(numeric_features))
print("Categorical features:", len(categorical_features))
print("Total:", len(numeric_features) + len(categorical_features))

Numerical features: 3
Categorical features: 16
Total: 19


#### Numerical preprocessing pipeline

In [14]:
numeric_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="median")
        ),
        (
            "scaler",
            StandardScaler()
        )
    ]
)

#### Categorical preprocessing pipeline

In [15]:
categorical_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="most_frequent"
            )
        ),
        (
            "onehot",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            )
        )
    ]
)

#### ColumnTransformer

In [16]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            numeric_pipeline,
            numeric_features
        ),
        (
            "cat",
            categorical_pipeline,
            categorical_features
        )
    ],
    remainder="drop"
)

In [19]:
# train data
preprocessor.fit(X_train)

,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details.",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The output of thetransformer is multiplied by these weights. Keys are transformer names,values the weights.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` and ``feature_name``

#### Train transform

In [20]:
X_train_processed = preprocessor.transform(
    X_train
)

In [21]:
# check
X_train_processed.shape

(5634, 46)

#### Test transform

In [22]:
X_test_processed = preprocessor.transform(
    X_test
)

In [23]:
# check
X_test_processed.shape

(1409, 46)

In [24]:
X_train_processed = (
    preprocessor.fit_transform(X_train)
)

X_test_processed = (
    preprocessor.transform(X_test)
)

#### Feature names

In [25]:
feature_names = (
    preprocessor.get_feature_names_out()
)

print("Number of processed features:",
      len(feature_names))

Number of processed features: 46


In [26]:
#check
feature_names

array(['num__tenure', 'num__MonthlyCharges', 'num__TotalCharges',
       'cat__gender_Female', 'cat__gender_Male', 'cat__SeniorCitizen_0',
       'cat__SeniorCitizen_1', 'cat__Partner_No', 'cat__Partner_Yes',
       'cat__Dependents_No', 'cat__Dependents_Yes',
       'cat__PhoneService_No', 'cat__PhoneService_Yes',
       'cat__MultipleLines_No', 'cat__MultipleLines_No phone service',
       'cat__MultipleLines_Yes', 'cat__InternetService_DSL',
       'cat__InternetService_Fiber optic', 'cat__InternetService_No',
       'cat__OnlineSecurity_No',
       'cat__OnlineSecurity_No internet service',
       'cat__OnlineSecurity_Yes', 'cat__OnlineBackup_No',
       'cat__OnlineBackup_No internet service', 'cat__OnlineBackup_Yes',
       'cat__DeviceProtection_No',
       'cat__DeviceProtection_No internet service',
       'cat__DeviceProtection_Yes', 'cat__TechSupport_No',
       'cat__TechSupport_No internet service', 'cat__TechSupport_Yes',
       'cat__StreamingTV_No', 'cat__StreamingTV_No

#### create DataFrame from Processed data  inspect

In [28]:
X_train_processed_df = pd.DataFrame(
    X_train_processed,
    columns=feature_names,
    index=X_train.index
)

X_train_processed_df.head()

,num__tenure,num__MonthlyCharges,num__TotalCharges,cat__gender_Female,cat__gender_Male,cat__SeniorCitizen_0,cat__SeniorCitizen_1,cat__Partner_No,cat__Partner_Yes,cat__Dependents_No,...,cat__StreamingMovies_Yes,cat__Contract_Month-to-month,cat__Contract_One year,cat__Contract_Two year,cat__PaperlessBilling_No,cat__PaperlessBilling_Yes,cat__PaymentMethod_Bank transfer (automatic),cat__PaymentMethod_Credit card (automatic),cat__PaymentMethod_Electronic check,cat__PaymentMethod_Mailed check
0,-1.281624,-1.164077,-0.995824,1.0,0.0,1.0,0.0,0.0,1.0,1.0,...,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0
1,0.061666,-0.264803,-0.179831,0.0,1.0,1.0,0.0,1.0,0.0,1.0,...,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0
2,-1.240918,-0.367672,-0.961467,0.0,1.0,1.0,0.0,1.0,0.0,1.0,...,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0
3,0.509429,-0.750942,-0.201222,0.0,1.0,1.0,0.0,1.0,0.0,1.0,...,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0
4,-1.240918,0.191470,-0.942379,1.0,0.0,1.0,0.0,1.0,0.0,1.0,...,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0


#### Numerical scaling verify

In [29]:
X_train_processed_df[
    [
        "num__tenure",
        "num__MonthlyCharges",
        "num__TotalCharges"
    ]
].mean()

num__tenure           -8.197600e-18
num__MonthlyCharges   -2.396222e-16
num__TotalCharges      2.837631e-17
dtype: float64

In [30]:
X_train_processed_df[
    [
        "num__tenure",
        "num__MonthlyCharges",
        "num__TotalCharges"
    ]
].std(ddof=0)

num__tenure            1.0
num__MonthlyCharges    1.0
num__TotalCharges      1.0
dtype: float64

#### Missing values check after preprocessing

In [31]:
np.isnan(
    X_train_processed
).sum()

np.int64(0)

In [32]:
#test
np.isnan(
    X_test_processed
).sum()

np.int64(0)

#### Shape validation

In [33]:
print(
    "Raw training shape:",
    X_train.shape
)

print(
    "Processed training shape:",
    X_train_processed.shape
)

print(
    "Raw test shape:",
    X_test.shape
)

print(
    "Processed test shape:",
    X_test_processed.shape
)

Raw training shape: (5634, 19)
Processed training shape: (5634, 46)
Raw test shape: (1409, 19)
Processed test shape: (1409, 46)


#### add Assertions

In [35]:
assert X_train.shape[1] == 19
assert X_train_processed.shape[0] == 5634
assert X_test_processed.shape[0] == 1409

assert (
    X_train_processed.shape[1]
    ==
    X_test_processed.shape[1]
)

assert not np.isnan(
    X_train_processed
).any()

assert not np.isnan(
    X_test_processed
).any()

print(
    "Preprocessing validation passed."
)

Preprocessing validation passed.


### Test

In [36]:
import sys

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

In [37]:
from src.features.preprocessing import build_preprocessor

In [38]:
test_preprocessor = build_preprocessor()

In [39]:
#actual test
X_train_test = test_preprocessor.fit_transform(
    X_train
)

X_test_test = test_preprocessor.transform(
    X_test
)

print("Training:", X_train_test.shape)
print("Testing :", X_test_test.shape)

Training: (5634, 46)
Testing : (1409, 46)


In [40]:
import numpy as np

print(
    "Train missing:",
    np.isnan(X_train_test).sum()
)

print(
    "Test missing:",
    np.isnan(X_test_test).sum()
)

Train missing: 0
Test missing: 0


In [41]:
from src.features.preprocessing import build_preprocessor